<a id='top'></a>
<img style="float: center;" src='https://github.com/STScI-MIRI/MRS-ExampleNB/raw/main/assets/banner1.png' alt="stsci_logo" width="1000px"/>

# NIRSpec MOS data reprocessing with the JWST pipeline
---

## Table of contents

* [1. Overview](#overview)
* [2. Observations](#observations)
* [3. JWST science calibration pipeline](#pipeline)
* [4. Related notebooks](#related)
* [5. Import libraries](#import)
* [6. Set up data directories](#setup)
* [7. Download UNCAL files: Query and retrieve from MAST archive](#data)
* [8. Stage1: `Detector1Pipeline`](#stage1)
* [9. Prepare for Stage 2](#stage2_prep)
* [10. Stage2: `Spec2Pipeline`](#stage2)
* [11. Prepare for Stage 3](#stage3_prep)
* [12. Stage3: `Spec3Pipeline`](#stage3)
* [13. `Extract1dStep`: Adjust 2D –> 1D extraction](#extract1d)
* [14. Compare spectra from MAST, pipeline reprocessing, and updated 1D extraction](#compare)
* [About this notebook](#about)

## 1. Overview <a id='overview'></a>
---

Data from JWST observations get processed by the JWST pipeline and made available in [MAST](https://mast.stsci.edu/search/ui/#/jwst). Outputs from any stage of the pipeline may be retrieved and reprocessed if desired.

There are some 
[known issues with NIRSpec MOS data products](https://jwst-docs.stsci.edu/known-issues-with-jwst-data/nirspec-known-issues/nirspec-mos-known-issues#gsc.tab=0)
and with the 
[JWST pipeline run on NIRSpec MOS data](https://jwst-docs.stsci.edu/jwst-calibration-pipeline-caveats/jwst-nirspec-mos-pipeline-caveats)
using default settings.

This example notebook demonstrates reprocessing NIRSpec MOS data from 
["JWST's First Deep Field"](https://webbtelescope.org/contents/news-releases/2022/news-2022-035), 
[ERO program 2736](https://www.stsci.edu/cgi-bin/get-proposal-info?observatory=JWST&id=2736)
that observed SMACS0723 with NIRCam, NIRSpec, and MIRI.
We retrieve the uncalibrated (`uncal`) data from MAST, rerun all 3 stages of the pipeline, 
and compare the final output spectrum to the products in MAST
for one galaxy #6355 at z = 7.665 (observed 13.0 billion years ago).

When rerunning the pipeline on this NIRSpec MOS data, we make two key adjustments to the following input parameters:
* outlier rejection of cosmic rays and snowballs: `Detector1Pipeline` jump step 
* extraction rows for 1D spectrum (`x1d`) from 2D spectrum (`s2d`): `extract1d` [`ystart` – `ystop`]
    * This works well now for most sources observed with the standard 3 shutter slitlets. 
    * Here we choose a source with a 5 shutter slitlet and make significant improvements.

## 2. Observations <a id='observations'></a>
--- 
[ERO 2736](https://www.stsci.edu/cgi-bin/get-proposal-info?observatory=JWST&id=2736) NIRSpec MOS observations:

* two identical observations: 7, 8
* (Confirmation images after TA: 20 groups NRSIRS2RAPID = 306 s)
* G235M & G395M
* 3-shutter nod x 2 integrations x 20 groups NRSIRS2 = 8841 s exposure time = 2.5 hours (in each grating and observation)

### Galaxies of interest
* **6355** z = 7.665 (13.0 Gyr ago)  
    * We'll look at this one in this notebook
    * excellent spectrum with bright lines
    * 5 shutters in a row (rather than standard 3), so 1D extraction can be improved significantly
    * not a multiple image of 10612 below at similar redshift
* 5144 z = 6.383
  
[featured in the press release](https://webbtelescope.org/contents/news-releases/2022/news-2022-035):
*  4590 z = 8.498 (13.1 Gyr ago)  
* 10612 z = 7.663 (13.0 Gyr ago)  
*  8140 z = 5.275 (12.6 Gyr ago)  
*  9922 z = 2.743 (11.3 Gyr ago)  

and presented and studied in papers including 
[Katz et al. 2023](https://ui.adsabs.harvard.edu/abs/2023MNRAS.518..592K), 
[Curti et al. 2023](https://ui.adsabs.harvard.edu/abs/2023MNRAS.518..425C), 
[Carnall et al. 2023](https://ui.adsabs.harvard.edu/abs/2023MNRAS.518L..45C)


## 3. JWST science calibration pipeline <a id='pipeline'></a>
---
**JWST Pipeline**
[JDox](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview) • 
[ReadtheDocs](https://jwst-pipeline.readthedocs.io) • 
[Github](https://github.com/spacetelescope/jwst) •
`pip install jwst`

Below we summarize the various pipeline stages and data products.

[**Data Products**](https://jwst-pipeline.readthedocs.io/en/latest/jwst/data_products/science_products.html)
(also see [JDox](https://jwst-docs.stsci.edu/getting-started-with-jwst-data/understanding-jwst-data-files/jwst-data-products))  

`uncal` (counts) –> `rate` (slopes) –> `cal` (extracted calibrated 2D spectra) –> `s2d` (rectified 2D spectra) –> `x1d` (1D spectra)

[Retrieve data from MAST](https://outerspace.stsci.edu/display/MASTDOCS/API+Advanced+Search)
(also see [ReadtheDocs](https://astroquery.readthedocs.io/en/latest/mast/mast_obsquery.html))

`uncal` uncalibrated data saved from every detector readout

[**JWST Pipeline Stages**](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing)
(also see [ReadtheDocs](https://jwst-pipeline.readthedocs.io/en/stable/jwst/pipeline/main.html))

[**Stage 1**](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing/calwebb_detector1):
[calwebb_detector1](https://jwst-pipeline.readthedocs.io/en/stable/jwst/pipeline/calwebb_detector1.html)
[`Detector1Pipeline`](https://jwst-pipeline.readthedocs.io/en/latest/api/jwst.pipeline.Detector1Pipeline.html)

Detector corrections for flat field, artifacts, etc., followed by ramp fitting to slopes (count rates)  
`rate` count rate (slope) averaged over multiple integrations, if available  
`rateints` count rates for each individual integration (saved in multiple extensions)

[**Stage 2**](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing/calwebb_spec2):
[calwebb_spec2](https://jwst-pipeline.readthedocs.io/en/latest/jwst/pipeline/calwebb_spec2.html)
[`Spec2Pipeline`](https://jwst-pipeline.readthedocs.io/en/latest/api/jwst.pipeline.Spec2Pipeline.html)

Extracted calibrated spectra for each source from each exposure (or association) and each detector, including any corrections for slit loss, path loss, and bar shadows.
Slits or exposures defined as background will have negative spectra.
Each source will be an extension in the file:  
`cal` / `calints` calibrated 2D (unrectified) spectra (flux vs. wavelength)  
`crf` / `crfints` cosmic ray flagged data, identical to `cal` products with updated DQ arrays  
`s2d` 2D spectra resampled (rectified) to a rectangle  
`x1d` 1D spectra extracted from rows of 2D spectrum

[**Stage 3**](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing/calwebb_spec3):
[calwebb_spec3](https://jwst-pipeline.readthedocs.io/en/latest/jwst/pipeline/calwebb_spec3.html)
[`Spec3Pipeline`](https://jwst-pipeline.readthedocs.io/en/latest/api/jwst.pipeline.Spec3Pipeline.html)

Extracted calibrated spectra for each source:  
`cal` / `calints` identical to Stage 2 outputs, repackaged for each source (multiple extensions for each exposure and detector)  
`crf` / `crfints` cosmic ray flagged data, identical to `cal` products with updated DQ arrays  
spectra combined from multiple exposures, including background subtraction:
`s2d` 2D spectra resampled (rectified) to a rectangle  
`x1d` 1D spectra extracted from rows of 2D spectrum

Note these filenames will have the same extensions (`cal`, `s2d`, `x1d`, etc.) but have different structures depending on which stage they are output from:
* Stage 2: one file from each exposure (or association) and detector; one extension for each source
* Stage 3: one file for each source; `cal`: one extension for each exposure (or association) and detector

## 4. Related notebooks <a id='related'></a>
---

* [NIRSpec MOS pipeline caveats](https://jwst-docs.stsci.edu/jwst-calibration-pipeline-caveats/jwst-nirspec-mos-pipeline-caveats)
and workaround notebooks for 
[general NIRSpec](https://github.com/spacetelescope/jwst-caveat-examples/tree/main/NIRSPEC_General)
data and
[NIRSpec MOS](https://github.com/spacetelescope/jwst-caveat-examples/tree/main/NIRSPEC_MOS)
* Notebooks running pipeline on [NIRSpec fixed slit data](coming soon) and
[BOTS data](https://github.com/exonik/JWebbinar2023-TSO)
* Notebooks running pipeline on simulated NIRSpec MOS data:
[JWebbinar7 (Oct 2021)](https://github.com/spacetelescope/jwebbinar_prep/blob/webbinar7/mos_session/jwebbinar7_nirspecmos.ipynb) •
[JDAT (2021)](https://github.com/spacetelescope/dat_pyinthesky/blob/main/jdat_notebooks/mos-spectroscopy/MOSspec_sv06_revised.ipynb) •
[JADES (June 2022)](https://github.com/eclake/NIRSpec_GTO_sims_STScI_reduction/blob/master/NIRSpec%20GTO%20IPS%20Simultions.ipynb)
* [Notebook processing ERO SMACS0723 NIRSpec MOS data](https://github.com/gbrammer/msaexp/blob/main/docs/examples/process-smacs0723.ipynb)
using the JWST pipeline and 
[msaexp](https://github.com/gbrammer/msaexp)
* [MOSViz notebook analyzing NIRSpec MOS spectra](https://spacetelescope.github.io/jdat_notebooks/notebooks/mos_spectroscopy_advanced/MOSspec_advanced.html)
* [JDAT: JWST Data Analysis Example Notebooks](https://github.com/spacetelescope/jdat_notebooks/tree/main/notebooks)

## 5. Import libraries <a id='import'></a>
---

In [ ]:
%load_ext pycodestyle_magic
%pycodestyle_on

In [ ]:
import json
import numpy as np
from glob import glob
from astropy.io import fits
# Nicer interactive output of JSON dictionaries
# from IPython.display import JSON

In [ ]:
# ---------- Set CRDS environment variables ----------
# Define CRDS paths *before* importing crds
# otherwise you'll get CrdsDownloadError: 
# Failed caching mapping files: Configured for server-less mode.  
# Skipping JSON RPC 'get_mapping_names'

# Define CRDS paths *before* importing jwst pipeline
# otherwise you'll get FileNotFoundError: [Errno 2] :
# No such file or directory: '$HOME/crds_cache/config/jwst/server_config'

import os
home = os.path.expanduser("~")
# os.environ['CRDS_CONTEXT'] = 'jwst_1210.pmap'  # if you want to set a specific context
os.environ["CRDS_PATH"] = os.path.join(home, "crds_cache")
os.makedirs(os.environ['CRDS_PATH'], exist_ok=True)
os.environ['CRDS_SERVER_URL'] = 'https://jwst-crds.stsci.edu'
print(f'CRDS cache location: {os.environ["CRDS_PATH"]}')

In [ ]:
import crds
try:
    print(f"Current Operational CRDS Context = {crds.get_default_context()}")
except Exception:
    print('CRDS server not found (are you offline?)')

In [ ]:
# Import JWST pipeline

import jwst
from jwst import datamodels
from jwst.pipeline import Detector1Pipeline   # calwebb_detector1
from jwst.pipeline import Spec2Pipeline       # calwebb_spec2
from jwst.pipeline import Spec3Pipeline       # calwebb_spec3
from jwst.extract_1d import Extract1dStep     # Extract1D Individual Step
from jwst.associations import asn_from_list   # create association file

# Note if you're using a development version,
# it might not report the correct version number here.
print("JWST Calibration Pipeline Version = {}".format(jwst.__version__))

In [ ]:
# To run faster on multiple cores in parallel
import multiprocessing as mp
print("Number of processors: ", mp.cpu_count())

In [ ]:
# To retrieve data from MAST
import astroquery
from astroquery.mast import Observations  # MAST
from astropy.table import unique  # Table
print('astroquery version', astroquery.__version__)

In [ ]:
# To plot and view results

import matplotlib  # as mpl
import matplotlib.pyplot as plt
# https://stackoverflow.com/questions/25426599/matplotlib-how-to-buffer-label-text
# import matplotlib.patheffects as pe

from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from astropy.visualization import simple_norm, ImageNormalize, AsinhStretch
# from astropy.visualization import LogStretch, LinearStretch, ManualInterval
from astropy.stats import sigma_clip  # sigma_clipped_stats, SigmaClip

%matplotlib inline
plt.rcParams.update({'font.size': 14})  # 18

In [ ]:
# Define colormap
# cmap = 'viridis'
# bad_color = 1, 0.7, 0.7
# bad_color = 'r'  # color used for nan values
# cmap = matplotlib.colormaps[cmap]
# cmap.set_bad(bad_color, 1.)

In [ ]:
def print_dict(d, indent=0):
    """
    Another way to view dictionary contents.

    Parameters:
    ----------
    d: dict
        Dictionary
    indent: int
        Number of indents (default = 0)

    Return:
    ------
    None
    """

    for key in d.keys():
        val = d[key]
        try:
            # keys = val.keys()
            print(' ' * indent, key)
            print_dict(val, indent+4)
        except Exception:
            if isinstance(val, list):
                for i, item in enumerate(val):
                    print(' ' * indent, '[%d]' % i)
                    print_dict(item, indent+4)
            else:
                print(' ' * indent, key, ':', d[key])

In [ ]:
def between(lo, x, hi):
    """
    Check if a value falls within a specified range.

    Parameters:
    ----------
    lo: float or int
        The lower bound of the range.
    x: float or int
        The value to check if it falls within the specified range.
    hi: float or int
        The higher bound of the range.

    Returns:
    -------
    int: 1 if the value x is between lo and hi (inclusive), 0 otherwise.
    """
    return (lo <= x) * (x <= hi)

## 6. Set up data directories <a id='setup'></a> 
---

In [ ]:
download_dir = 'data'  # directory name for files you download from MAST
os.makedirs(download_dir, exist_ok=True)  # create directory

In [ ]:
output_dir = 'reprocess'  # directory name for results from rerunning pipeline
os.makedirs(output_dir, exist_ok=True)  # create directory

In [ ]:
# If False, already ran the pipeline; just load and show the results
run_pipeline = False

Running the pipeline through all stages takes ~75 minutes on a Macbook Pro laptop:
* `Detector1` 25 minutes / 6 `UNCAL` –> `RATE`
* `Spec2` 25 minutes / 6 `RATE` –> `CAL`, `S2D`, `X1D`
* `Spec3`  25 minutes / 97 targets –> `CAL`, `S2D`, `X1D`

## 7. Download UNCAL files: Query and retrieve from MAST archive <a id='data'></a> 
---

[Astroquery](https://astroquery.readthedocs.io)
• [Downloading data](https://astroquery.readthedocs.io/en/latest/mast/mast_obsquery.html#downloading-data)
• [Observations Class](https://astroquery.readthedocs.io/en/latest/api/astroquery.mast.ObservationsClass.html)

[MAST JWST](https://mast.stsci.edu/search/ui/#/jwst)
• [API Advanced Search](https://outerspace.stsci.edu/display/MASTDOCS/API+Advanced+Search)
• [Products Field Descriptions](https://mast.stsci.edu/api/v0/_productsfields.html)

In [ ]:
# Helper function to download JWST files from MAST
def download_jwst_files(filenames, download_dir, mast_dir='mast:jwst/product'):
    """
    Helper function to download JWST files from MAST.

    Parameters:
    ----------
    filenames: list of str
        List of filenames to download.
    download_dir: str
        Directory where the files will be downloaded.
    mast_dir: str
        MAST directory containing JWST products.

    Returns:
    -------
    downloaded_files: list of str
        List of downloaded file paths.
    """
    # Download data
    downloaded_files = []
    os.makedirs(download_dir, exist_ok=True)
    for filename in filenames:
        filename = os.path.basename(filename)
        mast_path = os.path.join(mast_dir, filename)
        local_path = os.path.join(download_dir, filename)
        if os.path.exists(local_path):
            print(local_path, 'EXISTS')
        else:
            # Can let this command check if local file exists
            # However, it will delete it if it's there
            # and the wrong size (e.g., reprocessed)
            Observations.download_file(mast_path,   local_path=local_path)
        downloaded_files.append(local_path)

    return downloaded_files

In [ ]:
# Helper functions to select subset of files from list
def allin(elements, list_or_string):
    """
    Check if all elements are present in a list or string.

    Parameters:
    ----------
    elements: str or list
        Elements to check.
    list_or_string: list or str
        List or string to search for elements.

    Returns:
    -------
    bool: True if all elements are present, False otherwise.
    """
    if isinstance(elements, str):
        elements = elements.split()
    for element in elements:
        if element not in list_or_string:
            return False
    return True


def anyin(elements, list_or_string):
    """
    Check if any element is present in a list or string.

    Parameters:
    ----------
    elements: str or list
        Elements to check.
    list_or_string: list or str
        List or string to search for elements.

    Returns:
    -------
        bool: True if any element is present, False otherwise.
    """
    if isinstance(elements, str):
        elements = elements.split()
    for element in elements:
        if element in list_or_string:
            return True
    return False


# Select subset of files containing all search strings
def select_files(all_files, search_strings=[]):
    """
    Select a subset of files containing all search strings.

    Parameters:
    ----------
    all_files: list
        List of all files.
    search_strings: list
        List of search strings to match.

    Returns:
    -------
    chosen_files: list or str
        List of chosen files or a single chosen file if only one matches.
    """
    chosen_files = [file for file in all_files if allin(search_strings, file)]
    if len(chosen_files) == 1:
        chosen_files = chosen_files[0]
    return chosen_files

In [ ]:
# Observations.get_metadata("products")  # list all possible keyword filters

In [ ]:
# Find files already downloaded
uncal_files = glob(os.path.join(download_dir, '*_uncal.fits'))

products_list = []

# Download files from MAST if you haven't already.
# Could also let this run to check for all files.
# But the MAST query takes a minute.
if len(uncal_files) == 0:
    obs_table = Observations.query_criteria(obs_collection='JWST',
                                            # PID 2736: ERO SMACS0723
                                            proposal_id=2736,
                                            instrument_name='NIRSPEC/MSA',
                                            filters='g395m',
                                            dataproduct_type='spectrum')

    # All output products including images, catalogs, etc.
    products_list = Observations.get_product_list(obs_table)

    # UNCAL, RATE[INTS], CAL[INTS], I2D, S2D, X1D, ASN
    data_products = Observations.filter_products(
                                            products_list,
                                            productType='SCIENCE',
                                            productSubGroupDescription='UNCAL')

    # data_products  # list files retrieved from MAST

    # Only keep science files with full data;
    # not smaller files (confirmation images with fewer groups)
    file_sizes = unique(data_products, keys='size')['size']
    big_files = data_products[data_products['size'] > np.mean(file_sizes)]
    uncal_files = unique(big_files, keys='productFilename')['productFilename']
    # Observation 7 only, not Observation 8
    uncal_files = select_files(uncal_files, ['007'])

    uncal_files = download_jwst_files(uncal_files, download_dir)

In [ ]:
uncal_files = sorted(uncal_files)
# Remove TA confirmation files
# (they'll have EXP_TYPE = NRS_CONFIRM,
# whereas science exposures have NRS_MSASPEC)
uncal_files = [file for file in uncal_files
               if 'CONFIRM' not in fits.open(file)[0].header['EXP_TYPE']]

In [ ]:
# Final list of UNCAL files ready for Stage 1 processing
uncal_files

## 8. Stage 1 `Detector1Pipeline`: `uncal` –> `rate`  <a id='stage1'></a> 
---
`calwebb_detector1`
[JDox](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing/calwebb_detector1) •
[ReadtheDocs](https://jwst-pipeline.readthedocs.io/en/stable/jwst/pipeline/calwebb_detector1.html)

Detector corrections for flat-field, artifacts, etc., followed by ramp fitting to produce corrected count rate (slope) images.

This section took 25 minutes to process 6 UNCAL files (~4 minutes per UNCAL file) and generate RATE files.

### Input parameters
* [ramp fitting](https://jwst-pipeline.readthedocs.io/en/latest/jwst/ramp_fitting/arguments.html) (count rate slopes)  
* [jump step](https://jwst-pipeline.readthedocs.io/en/latest/jwst/jump/arguments.html) (including rejections of cosmic rays, snowballs, etc.)

CEERS NIRSpec reduction parameters to improve the jump step's rejection of cosmic rays and snowballs:
https://web.corral.tacc.utexas.edu/ceersdata/DR07/NIRSpec/README_NIRSpec_DR0.7.txt

In [ ]:
params_det1 = {}
params_det1['jump'] = {
    'expand_large_events': True,  # Turn on snowball flagging
    'expand_factor':  3.0,  # default 2
    'min_sat_area':  15.0,  # default 1
    'min_jump_area': 15.0,  # default 5
    # Run N times faster using N multiprocessors:
    'maximum_cores': 'half',  # integer, 'quarter', 'half', 'all'
}
params_det1['ramp_fit'] = {
    # Run N times faster using N multiprocessors:
    'maximum_cores': 'half',  # integer, 'quarter', 'half', 'all'
}

params_det1

### Run `Detector1Pipeline`

In [ ]:
if run_pipeline:
    for uncal_file in sorted(uncal_files):
        print('*' * 77)
        print("Applying basic detector-level corrections to " + uncal_file)

        det1_result = Detector1Pipeline.call(
            uncal_file,
            save_results=True,
            output_dir=output_dir,
            steps=params_det1)

<div class="alert alert-info">

**Note:** We run the pipeline using `.call()` instead of `.run()` to ensure we're using the latest default parameters from CRDS. For more information on `.call()` vs.  `.run()`, click the following links: [JDox](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline/running-the-jwst-science-calibration-pipeline#gsc.tab=0:~:text=Tso3Pipeline-,Running%20from%20within%20Python,-The%20pipeline%20can) • [ReadtheDocs](https://jwst-pipeline.readthedocs.io/en/latest/jwst/stpipe/call_via_run.html).

In [ ]:
# Print output result details:
# det1_result.__dict__  # view entire contents
# det1_result.meta.filename
# det1_result.data.shape

### Show rate file

In [ ]:
def show_MOS_rate(rate_file, slits_model=None,
                  save_plot=False, close_plot=False, integration=None,
                  cmap='viridis', bad_color=(1, 0.7, 0.7),
                  vmin=-0.003, vmax=0.022, show_colorbar=True,
                  title_prefix=None, title_path=False):
    """
    Parameters
    ----------
    rate_file: str
        Path to the rate/rateints FITS file (count rate data).
    slit_model: MultiSlitModel or None
        Slit models for all the sources.
        Can come from the CAL or S2D product datamodel (from Spec2Pipeline).
    save_plot: bool or str
        If True, save the plot as a PNG file.
        If a string is provided, it is used as the filename. Default is False.
    close_plot: bool
        If True, close the plot after displaying.
    integration: str or int or None
        If the data is 3D (rateints), define an integration to plot.
        If 'min', extracts the minimum slice of the cube for plotting.
        Default is None (assumes 2D data).
    cmap: str
        Colormap for the plot.
    bad_color: str or tuple
        Color of "bad" pixels (e.g., NaNs) in the colormap.
        Can be specified as a color name or RGB tuple (values 0 to 1).
    vmin/vmax: float
        Minimum/Maximum value for colormap range.
    show_colorbar: bool
        Show the figure colorbar? Default is True.
    title_prefix: str or None
        Prefix for the plot title.
    title_path: bool
        If True, include the full path of the FITS file in the plot title.
    """

    # Open the rate FITS file
    with fits.open(rate_file) as hdu_list:
        data = hdu_list['SCI'].data  # count rate data

        # Is the data 3D (_rateints.fits)?
        if integration == 'min':
            data = data.min(axis=0)  # min slice of cube
        elif integration is not None:
            data = data[integration]

    # Setup the figure and colorbar
    cmap = matplotlib.colormaps[cmap]
    cmap.set_bad(bad_color, 1.)  # color/opacity of bad pixels

    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())
    im = ax.imshow(data, origin='lower', cmap=cmap,
                   norm=norm, interpolation='nearest')
    ax.set_ylabel('Pixel Row')
    ax.set_xlabel('Pixel Column')

    if slits_model is not None:  # Draw slits and label source ids

        # add white outline to text below
        # path_effects=[pe.withStroke(linewidth=3, foreground="w", alpha=0.9)]
        path_effects = []  # no outline
        fontsize = 7
        color = 'w'

        # For each slitlet, draw the slit patch
        slit_patches = []
        for slit in slits_model.slits:
            slit_patch = Rectangle((slit.xstart, slit.ystart),
                                   slit.xsize, slit.ysize)
            slit_patches.append(slit_patch)

            y = slit.ystart + slit.ysize/2  # slit center location
            va = 'center'  # vertically centered text

            # Label the spectra on the left hand side for NRS1:
            if 'nrs1' in rate_file:
                x = slit.xstart
                ha = 'right'  # align test to the right
            else:  # Label the spectra on the right hand side for NRS2:
                x = slit.xstart + slit.xsize
                ha = 'left'  # align test to the left

            plt.text(x, y, slit.source_id, color=color, ha=ha, va=va,
                     fontsize=fontsize, path_effects=path_effects,
                     weight='bold')

        # plot the slit patches
        ax.add_collection(PatchCollection(slit_patches, ec='r', fc='None'))

    # plot title
    title = title_prefix + '  ' if title_prefix else ' '
    title += rate_file if title_path else os.path.basename(rate_file)

    if integration is not None:
        title = title.replace('rateints', 'rateints[%s]' % integration)

    plt.title(title)
    print(title)

    # https://stackoverflow.com/questions/18195758/set-matplotlib-colorbar-size-to-match-graph
    if show_colorbar:
        plt.subplots_adjust(left=0.05, right=0.85)
        units = 'DN/s'
        cbar_dx = 0.02
        cbar_ax = fig.add_axes([ax.get_position().x1+cbar_dx,
                                ax.get_position().y0, cbar_dx,
                                ax.get_position().height])
        cbar = fig.colorbar(im, label=units, cax=cbar_ax)
        cbar_ticks = cbar_ax.get_yticks()
        cbar_ticks = np.concatenate([cbar_ticks, [vmin, vmax]])
        cbar_ticks = np.compress(between(vmin, cbar_ticks, vmax),
                                 cbar_ticks)
        cbar_ticks = np.sort(cbar_ticks)
        cbar.set_ticks(cbar_ticks)
        # print(cbar_ticks)

    # Save the plot?
    if save_plot:
        if isinstance(save_plot, 'a.fits'):
            save_plot = rate_file.replace('fits', 'png')
            if integration is not None:
                save_plot = save_plot.replace('.png', '%s.png' % integration)
        plt.savefig(save_plot, dpi=200)
    if close_plot:
        plt.close()

#### The adjusted jump step parameters effectively reject cosmic rays and snowballs

In [ ]:
# Stage 1 Products
rate_files = sorted(glob(os.path.join(output_dir, '*_rate.fits')))
rate_files = select_files(rate_files, ['007'])  # Observation 7 only
rate_file = rate_files[-1]  # Show the last rate file as an example
rate_file

In [ ]:
show_MOS_rate(rate_file, title_prefix='REPROCESSED')

#### The default jump step parameters don't work as well

Compare to the data product in MAST generated using the pipeline with default parameters.

In [ ]:
# download_dir = 'data'
# Download the MAST data product (or load it if you've downloaded it already)
MAST_rate_file = download_jwst_files([rate_file], download_dir)[0]

In [ ]:
show_MOS_rate(MAST_rate_file, title_prefix='MAST')

## 9. Prepare for Stage 2 <a id='stage2_prep'></a> 
---
Below you will:
* Select which rate files to process
* Download/create the [association files](https://jwst-pipeline.readthedocs.io/en/latest/jwst/associations/overview.html) that define which rate files should be processed jointly. Here, we have the standard 3 nodded exposures, so we'll subtract them from each other.
* Download the MSA metafile defining the shutters (slitlets) and sources.

### Select which rate files to process

In [ ]:
# Let's just do Observation 7 (not 8).
# Can also select for '03103' in case
# the target acquisition data '03101' weren't weeded out above.
rate_files = sorted(glob(os.path.join(output_dir, '*_rate.fits')))
rate_files = select_files(rate_files, ['007', '03103'])
rate_files

### Association files

#### Do you have the association files already?

In [ ]:
# spec2_asn_files = glob(os.path.join(asn_dir, '*.json'))
spec2_asn_files = glob(os.path.join(output_dir, '*.json'))
spec2_asn_files = select_files(spec2_asn_files, ['o007', 'spec2'])
spec2_asn_files = sorted(spec2_asn_files)
spec2_asn_files

#### If needed, you could retrieve the association files from MAST

In [ ]:
# download ASN files
# asn_data_products = Observations.filter_products(
#                                 products_list,
#                                 productType='INFO',
#                                 productSubGroupDescription='ASN')

#### Here, we create the association files from scratch

For each detector, there are three rate files corresponding to the 3 nods (exposures). Each will have a turn being the signal labeled `science`, and the other two nods will be subtracted as `background`. 

In [ ]:
def create_spec2_asn_files(rate_files, output_dir):
    """
    Generate association (ASN) files for a set of nodded rate files.
    Each rate file will have an associated ASN file where it serves
    as the science exposure not background.

    Parameters
    ----------
    rate_files: list of str
        A list of strings representing the paths to the rate files.
    output_dir: str
        Path to the directory where the generated ASN files will be saved.

    Returns
    -------
    asn_files: list str
        A list of paths to the newly generated ASN files.
    """
    asn_files = []

    # Loop through the rate file list
    for science_index, science_rate_file in enumerate(rate_files):
        # Create empty ASN file
        product_name = os.path.basename(science_rate_file)
        asn_data = asn_from_list.asn_from_list(rate_files,
                                               product_name=product_name)

        # Add science and background exposures to the ASN file
        for i, member in enumerate(asn_data['products'][0]['members']):
            if i != science_index:
                member['exptype'] = 'background'

        # Name and save the ASN files
        asn_file = science_rate_file.replace('_rate.fits', '_asn.json')
        asn_file = os.path.join(output_dir, asn_file)
        asn_files.append(asn_file)
        print(asn_file)
        with open(asn_file, 'w') as outfile:
            name, serialized = asn_data.dump(format='json')
            outfile.write(serialized)

    return asn_files

In [ ]:
# Define subset of the rate files to use
rate_files_subset = select_files(rate_files, ['007', '03103'])  # observation 7
rate_files_subset = sorted(rate_files_subset)
rate_files_subset = [os.path.basename(file) for file in rate_files_subset]
rate_files_nrs1 = select_files(rate_files_subset, ['_nrs1_'])  # for NRS1
rate_files_nrs2 = select_files(rate_files_subset, ['_nrs2_'])  # for NRS2

In [ ]:
# Create association files for NRS1
spec2_asn_files_nrs1 = create_spec2_asn_files(rate_files_nrs1, output_dir)

In [ ]:
# Create association files for NRS2
spec2_asn_files_nrs2 = create_spec2_asn_files(rate_files_nrs2, output_dir)

In [ ]:
spec2_asn_files = spec2_asn_files_nrs1 + spec2_asn_files_nrs2

#### View contents of the association files

In [ ]:
# Print each association file
# followed by its contents, which define the science and background files
for asn_file in spec2_asn_files:
    print('-' * 33)
    print(os.path.basename(asn_file))
    with open(asn_file, 'r') as f_obj:
        asn_file_data = json.load(f_obj)

    for member in asn_file_data['products'][0]['members']:
        print(member['expname'], member['exptype'])

In [ ]:
# View full contents of one of the files (the last one)
print_dict(asn_file_data)

<div class="alert alert-info">
    
**Note**: Association files need to be in same directory as rate files.

### Retrieve the MSA METAFILE from MAST

The MSA metafile defines:  
* which MSA shutters (slitlets) are open, which sources they observe (if any), and whether they should be used for science or background in each dithered exposure
* the sources, including where they are (RA, Dec), and whether they should be treated as point sources (stellarity > 0.75) or extended (uniform illumination) for the path loss corrections

All MSA configurations may be stored in a single MSA metafile common to the observing program.

In [ ]:
for rate_file in rate_files:
    try:
        # Download MSA metafile
        msa_file = fits.getval(rate_file, 'MSAMETFL')
        download_jwst_files([msa_file], output_dir)
    except Exception:
        pass

Downloading URL https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:jwst/product/jw02736007001_01_msa.fits to reprocess/jw02736007001_01_msa.fits ... [Done]

In [ ]:
# Open MSA metafile
hdu_list = fits.open(rate_file)
msa_metafile = hdu_list[0].header['MSAMETFL']
msa_metafile = os.path.join(output_dir, msa_metafile)
msa_hdu_list = fits.open(msa_metafile)
msa_hdu_list.info()

If you wish to edit the MSA metafile, a detailed example notebook is forthcoming...

## 10. Stage 2 `Spec2Pipeline`: `rate` –> `cal`, `s2d`, `x1d`  <a id='stage2'></a> 
---
`calwebb_spec2`
[JDox](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing/calwebb_spec2) •
[ReadtheDocs](https://jwst-pipeline.readthedocs.io/en/latest/jwst/pipeline/calwebb_spec2.html)

Run on association files defining science and background slitlets for each of the 3 nod exposures: 
3 exposures x 2 detectors = 6 association files.

This section took 25 minutes to process 6 RATE files (~4 minutes per RATE) and generate CAL, S2D, and X1D files. The `Spec2Pipeline` takes ~7 minutes on a MacBook Pro laptop (with reference files already downloaded).

### Input parameters
* [resample](https://jwst-pipeline.readthedocs.io/en/latest/jwst/resample/arguments.html)
    * `weight_type`:
        * `ivm` pixel-based inverse read noise works best to reject outliers
        * `exptime` improves SNR, especially for bright objects (and will become the new default)

see https://jwst-docs.stsci.edu/jwst-calibration-pipeline-caveats/jwst-nirspec-mos-pipeline-caveats

In [ ]:
params_spec2 = {}

# This is the current default, but that will change, so set explicitly:
params_spec2['resample_spec'] = {
    # pixel-based inverse read noise (vs. 'exptime' single value exposure time)
    'weight_type': 'ivm'
}

params_spec2

### Run `Spec2Pipeline`

In [ ]:
if run_pipeline:
    for asn_file in spec2_asn_files:
        spec2_result = Spec2Pipeline.call(
            asn_file,
            save_results=True,
            output_dir=output_dir,
            steps=params_spec2
        )

### Label spectra in rate image

Draw boxes around extraction regions for each source in a RATE file, using slit information in the corresponding Spec2 CAL file. These boxes are large enough to accommodate the curved spectral traces. Neighboring boxes may overlap, even though the spectra do not. The spectra are subsequently rectified (straightened) in the S2D files, and 1D extracted in the X1D files.

In [ ]:
# Show the last rate file, as an example
# Label spectra using the corresponding cal file
rate_file = rate_files[-1]
cal_file = rate_file.replace('rate', 'cal')
print(cal_file)

In [ ]:
# Open the CAL file as a JWST datamodel
cal_model = datamodels.open(cal_file)
# cal_model.info(max_rows=99999)  # show all contents
# cal_data = cal_model.data + 0

In [ ]:
show_MOS_rate(rate_file, cal_model)  # , save_plot=True)

## 11. Prepare for Stage 3 <a id='stage3_prep'></a>
---

In [ ]:
# Create a subdirectory for the
# many output files (several for each extracted source)
spec3_output_dir = os.path.join(output_dir, 'spec3')
os.makedirs(spec3_output_dir, exist_ok=True)
spec3_output_dir

### Spec3 association file

#### If you already have a spec3 association file, list it here

In [ ]:
asn_files = glob(os.path.join(output_dir, '*.json'))
spec3_asn_file = select_files(asn_files, ['o007', 'spec3'])  # Observation 7
# Observation 8 is similar. Could do separately.
# Or could combine both observations.
spec3_asn_file

### Download the association file from MAST

In [ ]:
if len(spec3_asn_file) == 0:   # len(products_list) == 0
    # MAST query, if you haven't already. Takes a minute.
    obs_table = Observations.query_criteria(obs_collection='JWST',
                                            # PID 2736: ERO SMACS0723
                                            proposal_id=2736,
                                            instrument_name='NIRSPEC/MSA',
                                            filters='g395m',
                                            dataproduct_type='spectrum')

    # all output products including images, catalogs, etc.
    products_list = Observations.get_product_list(obs_table)

    # Filter the ASN files from the products list
    asn_data_products = Observations.filter_products(
        products_list, productType='INFO', productSubGroupDescription='ASN')

    asn_files = unique(asn_data_products,
                       keys='productFilename')['productFilename']
    spec3_asn_file = select_files(asn_files,
                                  ['o007', 'spec3'])  # Observation 7

    spec3_asn_file = download_jwst_files([spec3_asn_file], output_dir)[0]

### View contents of association file

In [ ]:
spec3_asn_file

In [ ]:
with open(spec3_asn_file) as f_obj:
    asn_data = json.load(f_obj)
    spec2_cal_files = []
    for member in asn_data['products'][0]['members']:
        if member['exptype'] == 'science':
            spec2_cal_file = member['expname']
            spec2_cal_files.append(spec2_cal_file)
            print(spec2_cal_file)

In [ ]:
# All science files will be combined in this stage.
# Nodded backgrounds were subtracted already by Spec2.
print_dict(asn_data)  # View full contents of one of the spec3 association file

## 11. Stage 3 `Spec3Pipeline`: combine exposures –> `s2d`, `x1d` for each target <a id='stage3'></a> 
---
`calwebb_spec3`
[JDox](https://jwst-docs.stsci.edu/jwst-science-calibration-pipeline-overview/stages-of-jwst-data-processing/calwebb_spec3) •
[ReadtheDocs](https://jwst-pipeline.readthedocs.io/en/latest/jwst/pipeline/calwebb_spec3.html)

This section took 25 minutes to process 97 sources (~15 seconds per source).

### Input parameters

In [ ]:
params_spec3 = {}
# Flag outlier bad pixels and cosmic rays in DQ array of each input image
params_spec3['outlier_detection'] = {'skip': False}
# pixel-based inverse read noise (vs. 'exptime' single value exposure time)
params_spec3['resample_spec'] = {'weight_type': 'ivm'}
# extraction rows based on expected source position
params_spec3['extract_1d'] = {'use_source_posn': 'True'}
params_spec3

In [ ]:
spec3_asn_file

### Run `Spec3Pipeline`

In [ ]:
if run_pipeline:
    spec3_result = Spec3Pipeline.call(
        spec3_asn_file,
        save_results=True,
        output_dir=spec3_output_dir,
        steps=params_spec3
    )

### Show spectrum result

In [ ]:
def show_MOS_spectrum(s2d_model, x1d_model, source_id=None, cmap='RdBu',
                      bad_color='w', ystart=None, ystop=None,
                      expand_wavelength_gap=True,
                      show_default_extraction=False,
                      default_extract_width=6):
    """
    Plot the 2D (S2D) and corresponding 1D (X1D) MOS spectra.

    Parameters:
    ----------
    s2d_model: MultiSlitModel
        Slit model(s) from an S2D product datamodel
        either for all the sources (from Spec2Pipeline)
        or from a single source (from Spec3Pipeline).
    x1d_model: MultiSlitModel
        Slit model(s) from an X1D product datamodel
        either for all the sources (from Spec2Pipeline)
        or from a single source (from Spec3Pipeline).
    source_id: int or None
        Source to inspect (may be left as None for 
        a single source from Spec3Pipeline).
    cmap: str
        Colormap for the plot.
    bad_color: str or tuple
        Color of "bad" pixels (e.g., NaNs) in the colormap.
        Can be specified as a color name or RGB tuple (values 0 to 1).
    ystart/ystop: float or None
        The 1D extraction region start and stop values.
    expand_wavelength_gap: bool
        Expand the wavelength gap to fill missing wavelength values?
        Default is True.
    default_extract_width: int
        If populated, displays the default 1D extraction region.
    """

    # 2D spectrum
    # s2d has all the objects; extract the one with source_id
    if 'slits' in list(s2d_model):
        source_ids = [slit.source_id for slit in s2d_model.slits]
        i_slit = source_ids.index(source_id)
        slit_model = s2d_model.slits[i_slit]
    else:  # s2d only has one object
        slit_model = s2d_model
        i_slit = 0

    s2d_data = slit_model.data + 0  # load and make copy
    # Replace zeros with nan where there is no data
    s2d_data = np.where(slit_model.err, s2d_data, np.nan)

    # 1D spectrum
    x1d_wave = x1d_model.spec[i_slit].spec_table.WAVELENGTH
    x1d_flux = x1d_model.spec[i_slit].spec_table.FLUX
    x1d_fluxerr = x1d_model.spec[i_slit].spec_table.FLUX_ERROR
    # fluxerr all nan?? pipeline bug
    if np.sum(np.isnan(x1d_fluxerr)) == len(x1d_fluxerr):
        # Replace zeros with nan where there is no data
        x1d_flux = np.where(x1d_flux, x1d_flux, np.nan)
    else:
        # Replace zeros with nan where there is no data
        x1d_flux = np.where(np.isnan(x1d_fluxerr), np.nan, x1d_flux)

    # Expand the wavelength array for the gap?
    if expand_wavelength_gap:
        # calculate differences between consecutive wavelengths and
        # find if there is a gap to fill
        dx1d_wave = x1d_wave[1:] - x1d_wave[:-1]
        igap = np.argmax(dx1d_wave)
        dx1d_max = np.max(dx1d_wave)
        dx_replace = (dx1d_wave[igap-1] + dx1d_wave[igap+1]) / 2.
        num_fill = int(np.round(dx1d_max / dx_replace))
        print("Expanding wavelength gap %.2f -- %.2f microns"
              % (x1d_wave[igap], x1d_wave[igap+1]))

        if num_fill > 1:  # There is a gap to fill
            wave_fill = np.mgrid[x1d_wave[igap]:
                                 x1d_wave[igap+1]:
                                 (num_fill+1)*1j]
            x1d_wave = np.concatenate([x1d_wave[:igap+1],
                                       wave_fill[1:-1],
                                       x1d_wave[igap+1:]])

            num_rows, num_waves = s2d_data.shape
            s2d_fill = np.zeros(shape=(num_rows, num_fill-1)) * np.nan
            s2d_data = np.concatenate([s2d_data[:, :igap+1],
                                       s2d_fill, s2d_data[:, igap+1:]], axis=1)

            x1d_fill = np.zeros(shape=(num_fill-1)) * np.nan
            x1d_flux = np.concatenate([x1d_flux[:igap+1],
                                       x1d_fill, x1d_flux[igap+1:]])
            x1d_fluxerr = np.concatenate([x1d_fluxerr[:igap+1],
                                          x1d_fill, x1d_fluxerr[igap+1:]])

    # Tick Labels
    wave_min, wave_max = x1d_wave[0], x1d_wave[-1]
    eps = 1e-7
    major_tick_interval = 0.2  # 0.5
    xtick_min = (np.ceil((wave_min - eps) / major_tick_interval)
                 * major_tick_interval)
    xticks = np.arange(xtick_min, wave_max, major_tick_interval)
    # xticks = np.append(xticks, [5.2])
    num_waves = len(x1d_wave)
    xtick_pos = np.interp(xticks, x1d_wave, np.arange(num_waves))
    xtick_labels = ['%.1f' % xtick for xtick in xticks]

    # Remove any tick labels that are too close and would overlap
    for i in range(len(xtick_pos)-1):
        dx = xtick_pos[i+1] - xtick_pos[i]
        if dx < 40:
            xtick_labels[i] = ''

    # for label in xtick_labels: print(label, ' ', end='')

    # Make Figure
    fig, (ax2d, ax1d) = plt.subplots(2, 1, figsize=(15, 8),
                                     height_ratios=[1, 3],
                                     squeeze=True, sharex=True)
    cmap = matplotlib.colormaps[cmap]
    cmap.set_bad(bad_color, 1.)

    # ymax = 2 * np.nanpercentile(s2d_data, 90)
    # ymin = -0.2 * ymax

    sigma_clipped_data = sigma_clip(s2d_data, sigma=5, maxiters=3)
    ymin = np.min(sigma_clipped_data)
    # ymin = 0.2 * ymin
    ymax = np.max(sigma_clipped_data)
    # center range about zero; but there may be some bias positive or negative
    # ymin = -ymax

    # Plot the rectified 2D spectrum
    # norm = simple_norm(s2dsci, 'linear', min_percent=2, max_percent=99.9)
    # norm = simple_norm(s2d_data, 'asinh',
    # min_percent=min_percent, max_percent=max_percent)
    norm = simple_norm(s2d_data, 'linear', min_cut=ymin, max_cut=ymax)
    print('s2d range:', norm.vmin, norm.vmax)

    ax2d.imshow(s2d_data, origin='lower', cmap=cmap,
                aspect='auto', norm=norm, interpolation='nearest')
    num_cross, num_dispersion = s2d_data.shape
    ny, nx = s2d_data.shape
    ax2d.yaxis.set_ticks_position('both')
    # ax2d.axhline((ny-1)/2., c='w', lw=0.5, alpha=0.5, ls='--')
    ax2d.set_ylabel("Pixel Row")

    if ystart:
        for y in ystart, ystop:
            # ax2d.axhline(y, c='w', lw=0.5, alpha=0.5, ls='--')
            ax2d.axhline(y, c='m', lw=0.5)
        ax2d.set_yticks([0, ystart, ystop, ny-1])
    else:
        ax2d.set_yticks([0, (ny-1)/2., ny-1])

    if show_default_extraction:
        extract_width = default_extract_width
        ystart0 = ny//2 - extract_width / 2
        ystop0 = ny//2 + extract_width / 2 - 1
        for y in ystart0, ystop0:
            ax2d.axhline(y, c='0.50', lw=0.5, ls='--', dashes=(10, 10))
        outline = ('Extraction region shown but not known: '
                   'dashed gray lines show default center region.')
        outline += ' May move with source position.'
        print(outline)

    # Plot the 1D extraction x1d vs. indices, same as s2d array above
    ax1d.axhline(0, c='0.50', lw=0.5, alpha=0.66, ls='-')
    ax1d.step(np.arange(num_waves), x1d_fluxerr, lw=0.5, c='r', alpha=0.66)
    ax1d.step(np.arange(num_waves), x1d_flux, lw=1)
    ax1d.set_xlim(0, num_waves)
    ax1d.yaxis.set_ticks_position('both')

    # ax1d.set_ylim(-2e-6, 2e-5)
    # ax1d.set_ylim(ymin, ymax)

    title = 'S2D  ' + os.path.basename(s2d_model.meta.filename)
    title += '  slit %d' % slit_model.slitlet_id
    title += '  source %d' % slit_model.source_id
    ax2d.set_title(title)
    ax1d.set_title('X1D Flux (Jy)')
    plt.xticks(xtick_pos, xtick_labels)
    plt.xlabel('wavelength (microns)')

In [ ]:
source_id = 6355  # Select source to inspect
# source_id = 5144  # Select source to inspect
# source_id = 4590  # Select source to inspect
# source_id = 10612  # Select source to inspect
# source_id = 8140  # Select source to inspect
# source_id = 9922  # Select source to inspect

In [ ]:
# output_dir = 'reprocess'
s2d_files = glob(os.path.join(spec3_output_dir, '*_s2d.fits'))
s2d_file = select_files(s2d_files, ['s%05d' % source_id])
s2d_file

In [ ]:
s2d_model = datamodels.open(s2d_file)
# s2d_model.info(max_rows=99999)  # show all contents
# s2d_data = s2d_model.data + 0

In [ ]:
# Load 1D extraction
x1d_file = s2d_file.replace('s2d', 'x1d')
print(x1d_file)
x1d_model = datamodels.open(x1d_file)
# x1d_model.info(max_rows=99999)  # show all contents

In [ ]:
show_MOS_spectrum(s2d_model, x1d_model, source_id, show_default_extraction=True)

Note the default extraction region misses the positive signal for this 5 shutter slitlet target and instead extracts negative signal from one of the nod subtractions.

## 13. `Extract1dStep`: Adjust 2D –> 1D extraction <a id='extract1d'></a>
---
[extract1d](https://jwst-pipeline.readthedocs.io/en/latest/jwst/extract_1d) • 
[Extract1dStep](https://jwst-pipeline.readthedocs.io/en/latest/api/jwst.extract_1d.Extract1dStep.html) • 
[Description](https://jwst-pipeline.readthedocs.io/en/latest/jwst/extract_1d/description.html#extraction-for-2d-slit-data) • 
[Editing JSON reference file](https://jwst-pipeline.readthedocs.io/en/latest/jwst/extract_1d/reference_files.html#editing-json-reference-file-format-for-non-ifu-data)

OPTIONAL to modify which rows in S2D (2D spectrum) are used for extraction to X1D (1D spectrum). 

**Note:** The X1D file now saves information about which rows were used in the `EXTRYSTR` and `EXTRYSTP` header keywords. These values are 1-indexed, not 0-indexed.

Define extraction region in `.json` file, then run `Extract1dStep`  
Output file will end in `_extract1dstep.fits`

`extract_width` takes priority over `ystart` and `ystop` for the extraction width,   
but `ystart` and `ystop` will still be used to define the centering of the extraction region in the cross-dispersion direction.

All of these values are zero-indexed integers. The start and stop limits are inclusive.

The default EXTRACT1D reference file is found in CRDS. The user can download this file from https://jwst-crds.stsci.edu, modify the contents, and then use this modified file in `Extract1dStep` by specifying this modified reference file with the `override_extract1d` option.

#### See also

* [Adjusting the extraction region](https://github.com/spacetelescope/jwst-caveat-examples/blob/main/NIRSPEC_General/nrs_recenter_extraction_workaround.ipynb)
* [MOSViz interactive extraction](https://spacetelescope.github.io/jdat_notebooks/notebooks/mos_spectroscopy_advanced/MOSspec_advanced.html)
* [Optimal extraction](https://spacetelescope.github.io/jdat_notebooks/notebooks/optimal_extraction/Spectral_Extraction-static.html)
* [msaexp optimal extraction](https://github.com/gbrammer/msaexp/blob/d7368560cf465a0828e942454cfcc0bbe6870278/msaexp/drizzle.py#L869C19-L869C19)


### The default `extract1d` parameter reference file may be retrieved from CRDS

In [ ]:
# This file can be found here: https://jwst-crds.stsci.edu
# default_extract1d_json_file = 'jwst_nirspec_extract1d_0003.json'

# crds_context = crds.get_default_context()

# Download the reference file, or find it in your cache
# default_extract1d_json_file = crds.getreferences(s2d_header,
#                                                reftypes=['extract1d'],
#                                                context=crds_context,
#                                                observatory='jwst')['extract1d']

In [ ]:
# List any extract1d JSON files created already
extract1d_json_files = glob('*extract1d*.json')
extract1d_json_files

### Here we create an `extract1d` JSON file from scratch

In [ ]:
extract1d_json_data = {'reftype': 'EXTRACT1D', 'instrument': 'NIRSPEC',
                       'telescope': 'JWST', 'exp_type': 'NRS_MSASPEC'}
extract1d_json_data['apertures'] = [{'id': 'ANY', 'region_type': 'target',
                                     'independent_var': 'pixel',
                                     'bkg_order': 0, 'dispaxis': 1,
                                     # F-flats now calibrated to this width
                                     'extract_width': 6}]

if run_pipeline:
    json_object = json.dumps(extract1d_json_data, indent=4)
    default_extract1d_json_file = 'jwst_nirspec_extract1d_default.json'
    with open(default_extract1d_json_file, "w") as outfile:
        outfile.write(json_object)

    # JSON(extract1d_json_data, expanded=True)
    print_dict(extract1d_json_data)

### Define the extraction region

In [ ]:
# ycenter = 15  # for most sources with standard 3 shutter slitlet nods
ycenter = 25  # here we have a larger 5 shutter slitlet extraction

# Use default extraction width
# extract_width = extract1d_json_data['apertures'][0]['extract_width']
# extract_width = 8  # nod self-subtraction for bright sources
extract_width = 6  # F-flats now calibrated to this width

ystart = ycenter - extract_width // 2
ystop = ystart + extract_width - 1
extract1d_json_data['apertures'][0]['ystart'] = ystart
extract1d_json_data['apertures'][0]['ystop'] = ystop
print(ystart, ystop)

In [ ]:
extract1d_json_data['apertures'][0]  # List extract1d parameter values

In [ ]:
# Create new JSON file
custom_extract1d_json_file = 'jwst_nirspec_extract1d_custom.json'
# if run_pipeline:
json_object = json.dumps(extract1d_json_data, indent=4)
with open(custom_extract1d_json_file, "w") as outfile:
    outfile.write(json_object)

custom_extract1d_json_file

In [ ]:
# Show contents of extract1d json file
with open(custom_extract1d_json_file) as f_obj:
    extract1d_json_data = json.load(f_obj)

# extract1d_json_data
# JSON(extract1d_json_data, expanded=True)
print_dict(extract1d_json_data)

### Run `Extract1d`

In [ ]:
# Run extract1d with parameters defined above
if run_pipeline:
    extract1d_result = Extract1dStep.call(
                                s2d_file,
                                use_source_posn=False,
                                apply_apcorr=False,
                                override_extract1d=custom_extract1d_json_file,
                                save_results=True, output_dir=spec3_output_dir)

# Output file ends in _extract1dstep (not _x1d);
# okay to save in same output directory
# jw02736-o007_s06355_nirspec_f290lp-g395m_extract1dstep.fits

In [ ]:
extract_x1d_file = s2d_file.replace('s2d', 'extract1dstep')
print(extract_x1d_file)
extract1d_result = datamodels.open(extract_x1d_file)

### Show the result

In [ ]:
# show_MOS_spectrum(s2d_model, x1d_model, source_id)
show_MOS_spectrum(s2d_model, extract1d_result,
                  ystart=ystart, ystop=ystop,
                  show_default_extraction=True)

## 14. Compare spectra from MAST, pipeline reprocessing, and updated 1D extraction <a id='compare'></a>
---

In [ ]:
s2d_file

In [ ]:
# Load original 2D and 1D spectra
x1d_file = s2d_file.replace('s2d', 'x1d')
s2d_model = datamodels.open(s2d_file)
x1d_model = datamodels.open(x1d_file)
x1d_wave = x1d_model.spec[0].spec_table.WAVELENGTH
x1d_flux = x1d_model.spec[0].spec_table.FLUX

In [ ]:
# Load new 1D extraction
extract1d_file = s2d_file.replace('s2d', 'extract1dstep')
extract1d_model = datamodels.open(extract1d_file)
extract1d_wave = extract1d_model.spec[0].spec_table.WAVELENGTH
extract1d_flux = extract1d_model.spec[0].spec_table.FLUX

In [ ]:
# Download default pipeline results from MAST for comparison
MAST_s2d_file = download_jwst_files([s2d_file], download_dir)[0]
MAST_x1d_file = download_jwst_files([x1d_file], download_dir)[0]
MAST_s2d_model = datamodels.open(MAST_s2d_file)
MAST_x1d_model = datamodels.open(MAST_x1d_file)
x1d_wave_MAST = MAST_x1d_model.spec[0].spec_table.WAVELENGTH
x1d_flux_MAST = MAST_x1d_model.spec[0].spec_table.FLUX

In [ ]:
# Define wavelength grid and ticks
num_waves = len(x1d_wave)
wave_min, wave_max = x1d_wave[0], x1d_wave[-1]
eps = 1e-7
# major_tick_interval = 0.5
major_tick_interval = 0.2
xtick_min = np.ceil((wave_min - eps) / major_tick_interval) * \
            major_tick_interval
xticks = np.arange(xtick_min, wave_max, major_tick_interval)
# xticks = np.append(xticks, [5.2])
num_waves = len(x1d_wave)
xtick_pos = np.interp(xticks, x1d_wave, np.arange(num_waves))
xtick_labels = ['%.1f' % xtick for xtick in xticks]

### Compare MAST (default pipeline) vs. pipeline reprocessing

The results are fairly similar. A few noise peaks are suppressed in the reprocessed data.

In [ ]:
fig, (ax2d, ax1d) = plt.subplots(2, 1, figsize=(15, 8),
                                 height_ratios=[1, 3],
                                 squeeze=True, sharex=True)

# plot the rectified 2D spectrum
# norm = simple_norm(s2dsci, 'linear', min_percent=10, max_percent=99.9)
# Replace zeros with nan where there is no data
s2d_data = np.where(s2d_model.err, s2d_model.data, np.nan)
norm = simple_norm(s2d_data, 'asinh', min_percent=2, max_percent=99.95)
cmap = matplotlib.colormaps['viridis']
cmap.set_bad((1, 0.8, 0.8), 1.)
ax2d.imshow(s2d_data, origin='lower', cmap=cmap,
            aspect='auto', norm=norm, interpolation='nearest')

ax1d.step(np.arange(num_waves), x1d_flux_MAST, lw=1, c='c', label='MAST')
ax1d.step(np.arange(num_waves), x1d_flux, lw=1,
          c='k', label='reprocessed', alpha=0.7)
# xlim = ax1d.get_xlim()
ax1d.set_xlim(0, num_waves)
ax1d.set_title('X1D Flux (Jy)')

plt.xticks(xtick_pos, xtick_labels)
plt.xlabel('wavelength (microns)')
plt.legend()

### Compare updated 1D extraction

For this source with a 5 shutter slitlet, the default extraction is centered, missing some of the spectrum, which is offset vertically. The manually adjusted 1D extraction rows yield significantly more flux.

**Sources with the standard 3 shutter slitlets would not show such significant improvement compared to the MAST default pipeline products.**

In [ ]:
fig, (ax2d, ax1d) = plt.subplots(2, 1, figsize=(15, 8), height_ratios=[1, 3],
                                 squeeze=True, sharex=True)

# plot the rectified 2D spectrum
# norm = simple_norm(s2dsci, 'linear', min_percent=10, max_percent=99.9)
# Replace zeros with nan where there is no data
s2d_data = np.where(s2d_model.err, s2d_model.data, np.nan)
norm = simple_norm(s2d_data, 'asinh', min_percent=2, max_percent=99.95)
cmap = matplotlib.colormaps['viridis']
cmap.set_bad((1, 0.8, 0.8), 1.)
ax2d.imshow(s2d_data, origin='lower', cmap=cmap,
            aspect='auto', norm=norm, interpolation='nearest')

ax1d.step(np.arange(num_waves), x1d_flux_MAST, lw=1,
          c='c', label='MAST', alpha=1)
ax1d.step(np.arange(num_waves), x1d_flux, lw=1,
          c='k', label='reprocessed',  alpha=0.5)
ax1d.step(np.arange(num_waves), extract1d_flux, lw=1,
          c='r', label='re-extracted', alpha=1)
# xlim = ax1d.get_xlim()
ax1d.set_xlim(0, num_waves)
ax1d.set_title('X1D Flux (Jy)')

ax2d.set_yticks([ystart, ystop])
for y in ystart, ystop:
    ax2d.axhline(y, c='r', alpha=1, lw=1, ls='-')

plt.xticks(xtick_pos, xtick_labels)
plt.xlabel('wavelength (microns)')
plt.legend()

# About this notebook <a id='about'></a>

**Authors:** Dan Coe (dcoe@stsci.edu) and Kayli Glidic with many contributions from others on the NIRSpec team, including Elena Manjavacas, Peter Zeidler, Melanie Clarke, James Muzerrole, Nikolay Nikolov, Chris Hayes, and Alaina Henry who designed the ERO NIRSpec observations.

**Updated On:** March 2024 running JWST pipeline version 1.13.4 with CRDS context jwst_1217.pmap

---

<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/notebooks/master/assets/stsci_pri_combo_mark_horizonal_white_bkgd.png" alt="Space Telescope Logo" width="200px"/>

[Top of Page](#top)   
